# Fixing resource-notation brittleness — consistency regularization

E5 showed the released CE model transfers to real tool-call *vocabulary* but is
sensitive to resource *notation*: on real tau2 it scores **90.8%** in the trained
`family:namespace/leaf` (slash) notation and **75.0%** re-rendered in all-colon.
This is the documented format-sensitivity of LLMs (Sclar et al., ICLR 2024).

**What failed and why.** Naive notation *augmentation* — retraining CE on
re-notated data (balanced or canonical-majority) — made it *worse*: real-tau2
accuracy became a seed lottery (50–81%) biased toward over-authorization. This
is exactly the documented failure mode: augmenting the training data for
*conventional fine-tuning* degrades **fine-grained** tasks, whereas using the
same augmentation for **consistency regularization** helps by a large margin
(Zheng et al., ACL 2021). Our confused-deputy task (distinguish `cust:0/…` from
`cust:5/…`) is fine-grained.

**The grounded fix (this notebook).** Keep CE anchored on the **canonical**
rendering (preserves the sharp 90.8% discrimination) and *add* a symmetric-KL
**consistency** term tying the model's verdict distribution on a re-notated view
of the same action to its canonical-view distribution (R-Drop style; Liang et
al., NeurIPS 2021). Notation-invariance is learned as *"match your own canonical
verdict,"* not as looser feature-matching. Flag: `--consistency-kl`.

**Deterministic complement.** Because we control how real logs map into the
schema, the deployment mitigation is **input canonicalization** — emit the
trained (slash) notation, i.e. the 90.8% column. Consistency-reg is the
model-side contribution that makes the model itself notation-robust.

**Comparison, all on the same held-out real tests** (tau2 slash+colon; Toucan
mixed, authorized-only): released baseline vs consistency-reg (default). The
balanced / canonical-majority negative variants are kept in Step 2, commented
out — uncomment to reproduce them. Every label comes from our verifier.

Runtime → T4/L4 → Run all. Consistency-reg is 3 seeds (~2–2.5 h; two forward
views per step). Download `augment_results.zip` at the end.

In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    !git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
os.chdir('/content/verifier-as-reward')
!git pull --ff-only
import torch; assert torch.cuda.is_available()
!PYTHONPATH=. python test_authority_verifier.py
!PYTHONPATH=. python make_expanded_train.py --train-seed 101 --train-traces-per-class 150 --val-seed 202 --val-traces-per-class 25

## Step 1 — build the augmented corpora (balanced + canonical-majority) and tests

In [ ]:
# balanced 4-way mix (the negative baseline)
!PYTHONPATH=. python -u augment_representation.py --in expanded_train.jsonl --out augmented_train.jsonl --seed 33
!PYTHONPATH=. python -u augment_representation.py --in expanded_val.jsonl   --out augmented_val.jsonl   --seed 34
# canonical-majority mix (70% trained slash, 30% split across other notations)
!PYTHONPATH=. python -u augment_representation.py --in expanded_train.jsonl --out augmented_train_cmaj.jsonl --seed 33 --canonical-frac 0.7
!PYTHONPATH=. python -u augment_representation.py --in expanded_val.jsonl   --out augmented_val_cmaj.jsonl   --seed 34 --canonical-frac 0.7
!PYTHONPATH=. python test_augment_representation.py
!PYTHONPATH=. python test_map_toucan.py

## Step 2 — train the grounded fix (consistency regularization), 3 seeds

`consistency` → `ckpt_consistency_seed{7,8,9}`: CE on the **canonical**
`expanded_train` + `--consistency-kl 1.0` (symmetric-KL tie to a re-notated
view). The `balanced` / `cmaj` negative variants are commented out below —
uncomment to reproduce them (they are already recorded in
`REALTRACE_FINDINGS.md`).

In [ ]:
# THE GROUNDED FIX — consistency regularization (CE on canonical + symmetric-KL to a re-notated view)
!for s in 7 8 9; do echo "=== consistency seed $s ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --ce-loss --balance-reward --prompt-style nl --consistency-kl 1.0 --steps 500 --batch-size 16 --lr 2e-5 --train-file expanded_train.jsonl --test-file expanded_val.jsonl --eval-every 50 --eval-max-actions 120 --seed $s --log-file training_log_consistency_seed$s.jsonl --save-dir ckpt_consistency_seed$s || break; done

# NEGATIVE RESULTS (already recorded) — uncomment to reproduce balanced / canonical-majority augmentation:
# !for s in 7 8 9; do echo "=== balanced seed $s ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --ce-loss --balance-reward --prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 --train-file augmented_train.jsonl --test-file augmented_val.jsonl --eval-every 50 --eval-max-actions 120 --seed $s --log-file training_log_balanced_seed$s.jsonl --save-dir ckpt_balanced_seed$s || break; done
# !for s in 7 8 9; do echo "=== cmaj seed $s ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --ce-loss --balance-reward --prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 --train-file augmented_train_cmaj.jsonl --test-file augmented_val_cmaj.jsonl --eval-every 50 --eval-max-actions 120 --seed $s --log-file training_log_cmaj_seed$s.jsonl --save-dir ckpt_cmaj_seed$s || break; done

## Step 3 — build the held-out REAL tests (never seen in training)

Two independent real logs. **tau2** (balanced confused-deputy) in both the
trained slash notation and the naive colon notation. **Toucan** (second,
independent source; authorized-only) rendered in **mixed native notations**
via `augment()` — each real trace gets a random delimiter scheme, labels
re-verified. All labels come from our verifier.

In [ ]:
from trace_benchmark import load_traces, write_jsonl
from augment_representation import renotate_trace, augment

# tau2 — balanced confused-deputy, slash (trained) + colon (naive)
!PYTHONPATH=. python map_tau_to_chain.py --seed 5   # writes real_trace_*.jsonl (slash)
slash = load_traces('real_trace_all.jsonl')
write_jsonl([renotate_trace(t, 'allcolon') for t in slash], 'real_trace_all_colon.jsonl')
print('tau2  slash:', slash[0]['actions'][0]['resource'],
      '| colon:', renotate_trace(slash[0], 'allcolon')['actions'][0]['resource'])

# Toucan — independent real source, authorized-only, MIXED native notations
!PYTHONPATH=. python map_toucan_to_chain.py --shards 1 --max-traj 200 --seed 5
toucan = load_traces('real_toucan_all.jsonl')
mixed, discarded = augment(toucan, seed=9)   # random delimiter scheme per trace, labels re-verified
write_jsonl(mixed, 'real_toucan_mixed.jsonl')
from collections import Counter
print('toucan mixed notations:', dict(Counter(t['_notation'] for t in mixed)),
      '| discarded (label changed):', discarded, '| n_traces:', len(mixed))

## Step 4 — evaluate released vs balanced vs canonical-majority on every held-out test

Robust in-process runner: each (model, test) is one subprocess; a non-zero exit
prints the stderr tail instead of silently dropping the row (the bug that hid the
real-test rows in the first run). tau2 is balanced (accuracy + false-authorize
matter); Toucan is authorized-only (its metric is false-refuse).

In [ ]:
import json
import os
import subprocess

RESULTS = 'results_augment.json'
json.dump({'backends': {}}, open(RESULTS, 'w'))

released = 'esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b'
real = ['real_trace_all.jsonl', 'real_trace_all_colon.jsonl', 'real_toucan_mixed.jsonl']

# released baseline, then whichever trained variants are present this session
jobs = [(released, ['benchmark_test.jsonl'] + real)]
for tag in ('consistency', 'balanced', 'cmaj'):
    for s in (7, 8, 9):
        ck = 'ckpt_%s_seed%d' % (tag, s)
        if os.path.isdir(ck):
            jobs.append((ck, ['benchmark_test.jsonl'] + real))

env = dict(os.environ, PYTHONPATH='.')
for model, files in jobs:
    for tf in files:
        if not os.path.exists(tf):
            print('SKIP', model, '::', tf, '(missing)')
            continue
        cmd = ['python', 'train_verifier_reward.py', '--eval-checkpoint',
               model, '--test-file', tf, '--merge-results', RESULTS]
        r = subprocess.run(cmd, env=env, capture_output=True, text=True)
        if r.returncode != 0:
            print('FAIL', model, '::', tf)
            print(r.stderr[-2000:])
        else:
            print('ok  ', model, '::', tf)

In [ ]:
import json
from stats import wilson_ci

d = json.load(open('results_augment.json'))
def pc(x):
    return 'n/a' if x is None else format(100 * x, '5.1f')
print('%-56s %4s %20s %6s %6s' % ('checkpoint / test', 'n', 'acc% [95% CI]', 'fauth', 'fref'))
for name in sorted(d['backends']):
    m = d['backends'][name]['metrics']
    n = m['n_actions']
    c = wilson_ci(round(m['accuracy'] * n), n)
    ci = '%5.1f [%.1f,%.1f]' % (100 * m['accuracy'], 100 * c[0], 100 * c[1])
    print('%-56s %4d %20s %6s %6s' % (
        name.replace('local:', '')[:56], n, ci,
        pc(m['false_authorize_rate']), pc(m['false_refuse_rate'])))
print('')
print('GOAL: consistency recovers tau2-slash ~90% AND lifts tau2-colon toward 85%+,')
print('stable across seeds (fauth stays low, no collapse). tau2 is the discriminating')
print('test; Toucan is authorized-only (watch fref). Deterministic fallback: canonicalize')
print('real inputs to the slash notation = the tau2-slash (~90.8%) column.')

In [ ]:
!zip -j augment_results.zip training_log_consistency_*.jsonl training_log_balanced_*.jsonl training_log_cmaj_*.jsonl results_augment.json real_toucan_mixed.jsonl real_trace_all_colon.jsonl 2>/dev/null
from google.colab import files
files.download('augment_results.zip')